In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import pandas as pd
import gdown
import numpy as np
import torch
import gc
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import ViTImageProcessor, ViTModel


In [ ]:
input_folders = os.listdir('/kaggle/input')
print("Daftar folder yang tersedia di /kaggle/input:")
print(input_folders)

In [ ]:
base_input = '/kaggle/input/datasets'
isi_datasets = os.listdir(base_input)
print(f"Isi folder datasets: {isi_datasets}")

In [ ]:
path_username = '/kaggle/input/datasets/fati22'
isi_fati22 = os.listdir(path_username)
print(f"Isi di dalam folder fati22: {isi_fati22}")

In [ ]:
dataset_path = '/kaggle/input/datasets/fati22/tokopedia-images-product'
try:
    total_gambar = len(os.listdir(dataset_path))
    print(f"Total gambar yang terdeteksi: {total_gambar:,}")
except FileNotFoundError:
    print("Path tidak ditemukan, pastikan penulisan path sudah benar.")
except Exception as e:
    print(f"Terjadi kesalahan: {e}")

In [ ]:
existing_images = {f.replace('.jpg', '') for f in os.listdir(dataset_path) if f.endswith('.jpg')}
print(f"Total ID Product yang ditemukan dalam folder: {len(existing_images):,}")

In [ ]:
file_id = '1CK5x5i5p1xrz4cKT_bJJY_N-9_C5zWuh'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'data_download.csv'
gdown.download(url, output, quiet=False)
df = pd.read_csv(output, sep=',', on_bad_lines='skip')

In [ ]:
df.head()

In [ ]:
len(existing_images)

In [ ]:
df_final = df[df["ID_Product"].astype(str).isin(existing_images)].copy()
df_final.shape

In [ ]:
duplikat = df[df.duplicated(subset=['ID_Product'], keep=False)]
print(f"Jumlah baris duplikat: {len(duplikat)}")

In [ ]:
df_final = df_final.drop_duplicates(subset=['ID_Product'], keep='first')
df_final.shape

In [ ]:
class TokopediaDataset(Dataset):
    def __init__(self, dataframe, img_dir, processor):
        self.df = dataframe
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        id_product = row['ID_Product']
        judul = row['Product_Name']
        
        # Sesuai skema jenius Anda: ID 1 -> 1.jpg
        img_name = f"{int(id_product)}.jpg"
        img_path = os.path.join(self.img_dir, img_name)
        
        try:
            image = Image.open(img_path).convert("RGB")
            inputs = self.processor(images=image, return_tensors="pt")
            pixel_values = inputs['pixel_values'].squeeze(0)
        except Exception:
            # Jika gambar tidak ada/rusak, beri tensor nol agar urutan tetap sinkron
            pixel_values = torch.zeros(3, 224, 224)
            
        return pixel_values, str(id_product), judul

In [ ]:
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
model = ViTModel.from_pretrained("google/vit-base-patch16-224").cuda()
model.eval()

In [ ]:
chunk_split = [
    #(0, 500000, 1),
    #(500000, 1000000, 2),
    (1000000, 1500000, 3),
    (1500000, 2000000, 4),
    #(2000000, 2500000, 5),
    #(2500000, 3000000, 6),
    #(3000000, None, 7) 
]

for start, end, chunk_num in chunk_split:
    if end:
        df_chunk = df_final.iloc[start:end].copy()
    else:
        df_chunk = df_final.iloc[start:].copy()
        
    dataset = TokopediaDataset(df_chunk, dataset_path, processor)
    dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=0)
    
    all_vectors, all_ids, all_juduls = [], [], []
    
    with torch.no_grad():
        for i, (imgs, ids, juduls) in enumerate(dataloader):
            imgs = imgs.cuda()
            outputs = model(pixel_values=imgs)
            embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_vectors.append(embeddings)
            all_ids.extend(ids)
            all_juduls.extend(juduls)
            if i % 200 == 0:
                print(f"Chunk {chunk_num} -> Batch {i} selesai, flush=True")
                
    df_output = pd.DataFrame({
        'ID_Product': all_ids,
        'Judul': all_juduls,
        'Embedding': np.vstack(all_vectors).tolist()
    })
    
    nama_file = f"chunk_{chunk_num}.parquet"
    df_output.to_parquet(nama_file, engine='pyarrow')
    print(f"Chunk {chunk_num} selesai")
    
    del df_chunk, dataset, dataloader, df_output, all_vectors, all_ids, all_juduls
    gc.collect()
    torch.cuda.empty_cache()